# QueryClarify in Google Colab

Run this notebook with **Runtime → Change runtime type → T4 GPU** when available. Add a `GITHUB_TOKEN` secret in Colab (Notebook settings → Secrets) and enable notebook access to it. The notebook clones `Trisha-Tyagi/QueryClarify`; keep Spider data in the repo or in Google Drive. The first model download can take a while; later runs can reuse the Hugging Face cache.

If Spider data is stored elsewhere in Drive, edit `DRIVE_DATA_DIR` below.

In [ ]:
from google.colab import drive, userdata
from pathlib import Path
import base64, os, subprocess, sys

repo_url = 'https://github.com/Trisha-Tyagi/QueryClarify.git'
PROJECT_DIR = Path('/content/QueryClarify')

if not (PROJECT_DIR / '.git').is_dir():
    token = userdata.get('GITHUB_TOKEN')
    if not token:
        raise ValueError('Add a GITHUB_TOKEN secret in Colab and enable notebook access to it.')
    env = os.environ.copy()
    env['GIT_TERMINAL_PROMPT'] = '0'
    env['GIT_CONFIG_COUNT'] = '1'
    env['GIT_CONFIG_KEY_0'] = 'http.https://github.com/.extraheader'
    credentials = base64.b64encode(f'x-access-token:{token}'.encode()).decode()
    env['GIT_CONFIG_VALUE_0'] = f'AUTHORIZATION: basic {credentials}'
    clone = subprocess.run(
        ['git', 'clone', repo_url, str(PROJECT_DIR)], env=env,
        capture_output=True, text=True
    )
    if clone.returncode:
        raise RuntimeError('Repository clone failed. Check the GitHub URL and token access; no credential was printed.')
if not (PROJECT_DIR / 'app_backend.py').is_file():
    raise FileNotFoundError('app_backend.py is missing from the GitHub checkout. Commit and push the latest QueryClarify project files, then rerun this cell.')

drive.mount('/content/drive')
os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
os.environ['HF_HUB_DISABLE_XET'] = '1'
DRIVE_DATA_DIR = Path('/content/drive/MyDrive/QueryClarify/spider_data')
REPO_DATA_DIR = PROJECT_DIR / 'spider_data'
DATA_DIR = REPO_DATA_DIR if (REPO_DATA_DIR / 'tables.json').is_file() else DRIVE_DATA_DIR
os.environ['QUERYCLARIFY_DATA_DIR'] = str(DATA_DIR)
print('Project:', PROJECT_DIR)
print('Spider data:', DATA_DIR)
print('GPU available:', __import__('torch').cuda.is_available())

In [ ]:
# Install the dependencies listed by the project.
%pip install -q -r requirements.txt

In [ ]:
from app_backend import load_schemas, load_pipeline_resources, execute_readonly

DATA_DIR = Path(os.environ['QUERYCLARIFY_DATA_DIR'])
if not (DATA_DIR / 'tables.json').is_file():
    raise FileNotFoundError(f'tables.json not found under {DATA_DIR}. Set QUERYCLARIFY_DATA_DIR to your Spider data folder.')
schemas = load_schemas(DATA_DIR)
print(f'Loaded {len(schemas)} database schemas')
print('Examples:', ', '.join(sorted(schemas)[:20]))

## Load the models

This loads the same models as the app. Set the model environment variables in this cell if you want to use different Hugging Face model IDs or local folders.

In [ ]:
LLM_NAME = os.getenv('QUERYCLARIFY_LLM_MODEL', 'Qwen/Qwen2.5-3B-Instruct')
EMBEDDING_NAME = os.getenv('QUERYCLARIFY_EMBEDDING_MODEL', 'all-MiniLM-L6-v2')
schemas, model, tokenizer, embedding_model, start_query, continue_query = load_pipeline_resources(
    str(DATA_DIR), LLM_NAME, EMBEDDING_NAME
)
print('Models loaded:', LLM_NAME, 'and', EMBEDDING_NAME)

## Ask a question

Choose a database ID from the list above, then edit the question and run the cell. If the system asks for clarification, answer in the next cell.

In [ ]:
DB_ID = sorted(schemas)[0]  # Change this to another database ID listed above.
QUESTION = 'Show the names of all singers.'

if DB_ID not in schemas:
    raise ValueError(f'Unknown DB_ID {DB_ID!r}. Choose an ID from the examples list.')
result = start_query(
    question=QUESTION, db_id=DB_ID, db=schemas[DB_ID],
    embedding_model=embedding_model, model=model, tokenizer=tokenizer
)
print('Status:', result.get('status'))
if result.get('status') == 'clarification_required':
    print('Clarification:', result.get('clarification_question'))
elif result.get('status') == 'completed':
    print(result['predicted_sql'])
else:
    print('Error:', result.get('message'))

In [ ]:
# Run this cell if the previous result requested clarification.
if result.get('status') == 'clarification_required':
    ANSWER = 'Replace this with your answer'
    result = continue_query(
        state=result['state'], clarification_answer=ANSWER,
        db=schemas[DB_ID], embedding_model=embedding_model,
        model=model, tokenizer=tokenizer
    )
    print('Status:', result.get('status'))
    if result.get('status') == 'completed':
        print(result['predicted_sql'])
    else:
        print(result.get('message', result.get('clarification_question')))

In [ ]:
# Execute generated SQL using the project's read-only SQLite helper.
if result.get('status') == 'completed' and result.get('predicted_sql'):
    columns, rows, truncated = execute_readonly(
        DATA_DIR / 'database', DB_ID, result['predicted_sql']
    )
    from IPython.display import display, HTML
    import pandas as pd
    display(pd.DataFrame(rows, columns=columns))
    print(f'{len(rows)} rows returned' + (' (first 500 shown)' if truncated else ''))
else:
    print('Generate a completed query first.')